In [0]:
# ============================================================
# Medallion Architecture — Bronze → Silver + Quarantine
# hrms_employee_dimension Pipeline
# ============================================================

# COMMAND ----------
# STEP 1: Import Components and Establish Environment Context
# ──────────────────────────────────────────────────────────
# Importing all required PySpark modules.
# Window is needed for duplicate ranking logic.
# current_date() is used to compute tenure at pipeline run time.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Medallion_Architecture_EmployeeDimension_Silver_Quarantine") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sql("CREATE SCHEMA IF NOT EXISTS hackathon_ltm.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS hackathon_ltm.quarantine")

print("✅ STEP 1 COMPLETE — Spark session initialized. Target schemas verified.")


# COMMAND ----------
# STEP 2: Ingest Raw Records from Bronze Delta Table
# ──────────────────────────────────────────────────
# Reading the raw employee data as-is from the Bronze layer.
# No transformations at this stage — Bronze is always kept immutable.

bronze_table_name = "hackathon_ltm.bronze.hrms_employee_dimension"
df_bronze = spark.read.table(bronze_table_name)

bronze_count = df_bronze.count()
print(f"✅ STEP 2 COMPLETE — Bronze records ingested: {bronze_count}")


# COMMAND ----------
# STEP 3: Silver Layer Transformations — Cleansing & Standardization
# ──────────────────────────────────────────────────────────────────
# All transformations below operate on intermediate columns first,
# then the original columns are replaced at the end of this block.
# This ensures the original Bronze values are preserved in Quarantine.

# ── TRANSFORMATION 1: Trim whitespace from all string columns ──────
# WHY: The 'department' column had " Data Engineering" (leading space)
#      in 1 row. Untrimmed values cause silent GROUP BY mismatches in
#      Gold layer aggregations and duplicate entries in Power BI visuals.

string_cols = [f.name for f in df_bronze.schema.fields if str(f.dataType) == "StringType()"]
df_transformed = df_bronze
for col_name in string_cols:
    df_transformed = df_transformed.withColumn(col_name, F.trim(F.col(col_name)))

# ── TRANSFORMATION 2: Uppercase employee_id ────────────────────────
# WHY: employee_id is the primary key used to JOIN with sp_certification
#      and other fact tables. Inconsistent casing causes join failures.
#      Standardizing to UPPER ensures reliable cross-table lookups.

df_transformed = df_transformed.withColumn(
    "employee_id",
    F.upper(F.col("employee_id"))
)

# ── TRANSFORMATION 3: Title Case for human-readable string columns ─
# WHY: Columns like employee_name, department, designation, location,
#      and manager_name appear directly in Power BI reports and dashboards.
#      Mixed casing ("data engineering" vs "Data Engineering") creates
#      duplicate slicers and incorrect groupings in visuals.

for col_name in ["employee_name", "department", "designation", "location",
                 "manager_name", "employment_type", "status"]:
    df_transformed = df_transformed.withColumn(col_name, F.initcap(F.col(col_name)))

# ── TRANSFORMATION 4: Parse joining_date string → DateType ────────
# WHY: joining_date arrives as a raw string from Bronze. It must be
#      cast to a true Spark DateType so we can compute tenure_years,
#      joining_year, and joining_month as proper date-based KPI columns.
#      The standard format in this dataset is "yyyy-MM-dd".
#      Wrong formats / impossible dates (Feb 30) will parse to NULL
#      and are caught by Quarantine Rule 3 and Rule 4 below.

df_transformed = df_transformed.withColumn(
    "joining_date_parsed",
    F.try_to_date(F.col("joining_date"), "yyyy-MM-dd")
)

print("✅ STEP 3 COMPLETE — All Silver transformations applied.")


# COMMAND ----------
# STEP 4: Duplicate Detection & Separation
# ────────────────────────────────────────
# TWO types of duplicates exist in this dataset:
#
#   TYPE A — Exact duplicate employee_id (same ID, same data):
#     3 pairs found: E10015, E10082, E10490.
#     These are straightforward re-ingestion duplicates.
#     → Keep rank-1, quarantine rank-2+.
#
#   TYPE B — Same email across different employee_ids:
#     3 pairs found: (E10200, E10201), (E10300, E10301), (E10400, E10401).
#     Same name + adjacent IDs + same email = likely data entry duplicate
#     where a new ID was accidentally created for the same person.
#     → BOTH records flagged and sent to Quarantine for manual review.
#     → We do NOT auto-keep one, because we cannot determine which
#       employee_id is the canonical one without HR confirmation.

# ── TYPE A: Duplicate employee_id ──────────────────────────────────
df_transformed = df_transformed.withColumn("_row_id", F.monotonically_increasing_id())

window_spec = Window \
    .partitionBy("employee_id") \
    .orderBy(F.col("_row_id").asc())

df_ranked = df_transformed.withColumn("_row_rank", F.row_number().over(window_spec))

df_deduped_a  = df_ranked.filter(F.col("_row_rank") == 1).drop("_row_rank", "_row_id")
df_dup_emp_id = df_ranked.filter(F.col("_row_rank") > 1).drop("_row_rank", "_row_id")

# ── TYPE B: Duplicate email across different employee_ids ───────────
email_window = Window.partitionBy("email")
df_deduped_a = df_deduped_a.withColumn(
    "_email_count",
    F.count("employee_id").over(email_window)
)

# Both records go to quarantine — flagged for manual review
df_dup_email  = df_deduped_a.filter(
    F.col("email").isNotNull() & (F.col("_email_count") > 1)
).drop("_email_count")

# Only truly unique email records move forward
df_deduped = df_deduped_a.filter(
    F.col("email").isNull() | (F.col("_email_count") == 1)
).drop("_email_count")

type_a_count = df_dup_emp_id.count()
type_b_count = df_dup_email.count()
print(f"✅ STEP 4 COMPLETE — Duplicates separated:")
print(f"   Type A (duplicate employee_id)            : {type_a_count} rows")
print(f"   Type B (duplicate email, different emp_id): {type_b_count} rows")


# COMMAND ----------
# STEP 5: Quarantine Rule Evaluation & Constraint Mapping
# ───────────────────────────────────────────────────────
# Rules are applied AFTER deduplication so each unique row is evaluated
# independently against data quality constraints.
#
#   RULE 1 — Missing Primary Key (employee_id null or empty)
#     A row without employee_id cannot join to any fact table.
#
#   RULE 2 — Missing Email
#     2 rows have NULL email. Email is a key contact field used for
#     employee lookup and reporting. Flagged for HR team to fill.
#
#   RULE 3 — Corrupted joining_date Format
#     E10035 has "15/02/2020" (DD/MM/YYYY) — wrong format.
#     to_date("yyyy-MM-dd") returns NULL for this row.
#     Separating it avoids polluting tenure KPIs with NULL tenure.
#
#   RULE 4 — Impossible joining_date (non-existent date)
#     5 rows have "2023-02-30" (Feb 30 does not exist in any year).
#     to_date() returns NULL for these. These are data entry errors
#     that need correction before entering Silver.

df_evaluated = df_deduped.withColumn(
    "_failed_constraints",
    F.array(
        # Rule 1 — Missing Primary Key
        F.when(
            F.col("employee_id").isNull() | (F.col("employee_id") == ""),
            "ERR: Missing Primary Key (employee_id)"
        ),

        # Rule 2 — Missing Email
        F.when(
            F.col("email").isNull() | (F.col("email") == ""),
            "ERR: Missing email"
        ),

        # Rule 3 — Corrupted joining_date format
        # joining_date has a value, but to_date() returned NULL
        # meaning it could not be parsed (e.g. DD/MM/YYYY format)
        F.when(
            F.col("joining_date").isNotNull() &
            F.col("joining_date_parsed").isNull(),
            "ERR: Corrupted joining_date (wrong format or impossible date)"
        )
    )
)

# array_compact strips NULLs — only real error strings remain
# Clean rows  → [] (size 0) → Silver
# Dirty rows  → ["ERR: ..."] (size > 0) → Quarantine
df_evaluated = df_evaluated.withColumn(
    "_failed_constraints",
    F.array_compact(F.col("_failed_constraints"))
    # Spark < 3.4 alternative:
    # F.expr("filter(_failed_constraints, x -> x is not null)")
)

print("✅ STEP 5 COMPLETE — Quarantine constraint rules evaluated.")


# COMMAND ----------
# STEP 6: Split Data Streams — Silver vs. Quarantine
# ──────────────────────────────────────────────────
# Three quarantine streams are merged into one unified quarantine table:
#   Stream 1 — Constraint failures (missing ID, email, bad date)
#   Stream 2 — Type A duplicates (duplicate employee_id)
#   Stream 3 — Type B duplicates (duplicate email, different employee_id)
# Each stream carries its own _failed_constraints tag for audit clarity.

# 6.1 — Rows that failed constraint checks
df_quarantine_constraints = df_evaluated.filter(
    F.size(F.col("_failed_constraints")) > 0
)

# 6.2 — Clean Silver rows (passed all constraint checks)
#       Drop the internal helper columns before writing
df_silver_batch = df_evaluated \
    .filter(F.size(F.col("_failed_constraints")) == 0) \
    .drop("joining_date", "_failed_constraints") \
    .withColumnRenamed("joining_date_parsed", "joining_date")

# 6.3 — Tag Type A duplicates
df_dup_emp_id_tagged = df_dup_emp_id.withColumn(
    "_failed_constraints",
    F.array(F.lit("ERR: Duplicate employee_id"))
)

# 6.4 — Tag Type B duplicates
df_dup_email_tagged = df_dup_email.withColumn(
    "_failed_constraints",
    F.array(F.lit("ERR: Duplicate email across different employee_id — manual review required"))
)

# 6.5 — Union all three quarantine streams
df_quarantine_batch = df_quarantine_constraints \
    .unionByName(df_dup_emp_id_tagged) \
    .unionByName(df_dup_email_tagged)

print("✅ STEP 6 COMPLETE — Data streams split into Silver and Quarantine.")


# COMMAND ----------
# STEP 7: Add Derived Columns to Silver (for Gold KPIs)
# ─────────────────────────────────────────────────────
# These columns are computed from existing Silver fields and added
# directly to the Silver table so the Gold layer only needs simple
# GROUP BY aggregations — no date arithmetic or CASE logic in Gold.
#
# DERIVED COLUMN 1 — tenure_years
#   Formula : DATEDIFF(today, joining_date) / 365.25
#   KPI use : Average tenure per department, per grade, per location.
#             Rounded to 2 decimal places for readability.
#
# DERIVED COLUMN 2 — tenure_band
#   Formula : Bucket tenure_years into 5 bands
#   KPI use : Headcount distribution by seniority band.
#             Useful for attrition analysis and workforce planning.
#             Bands: 0-1 yr | 1-3 yrs | 3-5 yrs | 5-10 yrs | 10+ yrs
#
# DERIVED COLUMN 3 — joining_year
#   Formula : YEAR(joining_date)
#   KPI use : Year-over-year hiring trend chart in Power BI.
#
# DERIVED COLUMN 4 — joining_month
#   Formula : DATE_FORMAT(joining_date, 'yyyy-MM')
#   KPI use : Monthly hiring seasonality analysis.
#             Answers "which months do we hire the most?"
#
# DERIVED COLUMN 5 — is_manager
#   Formula : grade LIKE 'M%' → True / False
#   KPI use : Manager vs Individual Contributor headcount ratio.
#             Useful for org structure KPIs and span-of-control reports.
#
# DERIVED COLUMN 6 — is_active
#   Formula : status = 'Active' → True / False
#   KPI use : Active headcount count, attrition rate calculation.
#             Filters in Power BI can use this boolean directly.

df_silver_batch = df_silver_batch \
    .withColumn(
        "tenure_years",
        F.round(F.datediff(F.current_date(), F.col("joining_date")) / 365.25, 2)
    ) \
    .withColumn(
        "tenure_band",
        F.when(F.col("tenure_years") < 1,  "0-1 yr")
         .when(F.col("tenure_years") < 3,  "1-3 yrs")
         .when(F.col("tenure_years") < 5,  "3-5 yrs")
         .when(F.col("tenure_years") < 10, "5-10 yrs")
         .when(F.col("tenure_years") >= 10, "10+ yrs")
         .otherwise(None)
    ) \
    .withColumn(
        "joining_year",
        F.year(F.col("joining_date"))
    ) \
    .withColumn(
        "joining_month",
        F.date_format(F.col("joining_date"), "yyyy-MM")
    ) \
    .withColumn(
        "is_manager",
        F.when(F.col("grade").startswith("M"), True).otherwise(False)
    ) \
    .withColumn(
        "is_active",
        F.when(F.col("status") == "Active", True).otherwise(False)
    )

print("✅ STEP 7 COMPLETE — 6 derived KPI columns added to Silver.")


# COMMAND ----------
# STEP 8: Diagnostics — Confirm Split Before Writing
# ──────────────────────────────────────────────────

silver_count      = df_silver_batch.count()
quarantine_count  = df_quarantine_batch.count()

print(f"\n{'='*60}")
print(f"  Bronze  (total)                  : {bronze_count}")
print(f"  Silver  (clean)                  : {silver_count}")
print(f"  Quarantine (constraint failures) : {df_quarantine_constraints.count()}")
print(f"  Quarantine (duplicate emp_id)    : {type_a_count}")
print(f"  Quarantine (duplicate email)     : {type_b_count}")
print(f"  Quarantine (total)               : {quarantine_count}")
print(f"  Accounted for                    : {silver_count + quarantine_count} / {bronze_count}")
print(f"{'='*60}\n")

# Error distribution in quarantine
print("Quarantine error breakdown:")
df_quarantine_batch \
    .select(F.explode(F.col("_failed_constraints")).alias("error")) \
    .groupBy("error") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(truncate=False)

# Silver derived columns preview
print("Silver derived columns preview (5 rows):")
df_silver_batch.select(
    "employee_id", "employee_name", "joining_date",
    "tenure_years", "tenure_band", "joining_year",
    "joining_month", "is_manager", "is_active"
).show(5, truncate=False)

print("✅ STEP 8 COMPLETE — Diagnostics verified.")


# COMMAND ----------
# STEP 9: Write to Target Delta Tables
# ─────────────────────────────────────

# 9.1 — Write clean records to Silver
silver_table_employee_dimension = "hackathon_ltm.silver.hrms_employee_dimension"

df_silver_batch.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table_employee_dimension)

print(f"✅ Silver table written → {silver_table_target} ({silver_count} rows)")

# 9.2 — Write all quarantine records (all 3 streams) to Quarantine
quarantine_table_employee_dimension = "hackathon_ltm.quarantine.hrms_employee_dimension"

df_quarantine_batch.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(quarantine_table_employee_dimension)

print(f"✅ Quarantine table written → {quarantine_table_employee_dimension
      } ({quarantine_count} rows)")

print(f"\n{'='*60}")
print(f"  ✅ PIPELINE COMPLETE")
print(f"  Clean      → {silver_table_target}")
print(f"  Quarantine → {quarantine_table_employee_dimension}")
print(f"{'='*60}")

✅ STEP 1 COMPLETE — Spark session initialized. Target schemas verified.
✅ STEP 2 COMPLETE — Bronze records ingested: 493
✅ STEP 3 COMPLETE — All Silver transformations applied.
✅ STEP 4 COMPLETE — Duplicates separated:
   Type A (duplicate employee_id)            : 3 rows
   Type B (duplicate email, different emp_id): 6 rows
✅ STEP 5 COMPLETE — Quarantine constraint rules evaluated.
✅ STEP 6 COMPLETE — Data streams split into Silver and Quarantine.
✅ STEP 7 COMPLETE — 6 derived KPI columns added to Silver.

  Bronze  (total)                  : 493
  Silver  (clean)                  : 476
  Quarantine (constraint failures) : 8
  Quarantine (duplicate emp_id)    : 3
  Quarantine (duplicate email)     : 6
  Quarantine (total)               : 17
  Accounted for                    : 493 / 493

Quarantine error breakdown:
+--------------------------------------------------------------------------+-----+
|error                                                                     |count|
+-----

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ── Read the Silver table ──────────────────────────────────────────
table_name = "hackathon_ltm.silver.hrms_employee_dimension"
df = spark.read.table(table_name)

# ── Verify before updating — check current distribution ───────────
print("BEFORE update — employment_type distribution:")
df.groupBy("employment_type").count().orderBy("employment_type").show()

# ── Apply the fix: Replace 'Trainee' → 'Full-time' ────────────────
df_fixed = df.withColumn(
    "employment_type",
    F.when(F.col("employment_type") == "Trainee", "Full-time")
     .otherwise(F.col("employment_type"))
)

# ── Overwrite the Silver Delta table with the corrected data ───────
df_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "false") \
    .saveAsTable(table_name)

# ── Verify after updating ──────────────────────────────────────────
df_verify = spark.read.table(table_name)
print("AFTER update — employment_type distribution:")
df_verify.groupBy("employment_type").count().orderBy("employment_type").show()

print("✅ Done — 'Trainee' successfully replaced with 'Full-time'.")

BEFORE update — employment_type distribution:
+---------------+-----+
|employment_type|count|
+---------------+-----+
|       Contract|   39|
|      Full-time|  371|
|        Trainee|   66|
+---------------+-----+

AFTER update — employment_type distribution:
+---------------+-----+
|employment_type|count|
+---------------+-----+
|       Contract|   39|
|      Full-time|  437|
+---------------+-----+

✅ Done — 'Trainee' successfully replaced with 'Full-time'.
